In [23]:
import json
import os

parent_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
dataset_dir = f'{parent_dir}/dataset'

with open(f'{dataset_dir}/correct_ids/tracks_filled.json') as f:
    tracks = json.load(f)

In [24]:
texts = list()

for track in tracks:
    texts.append(track['lyrics'])

In [25]:
# count = 0
# for i in range(len(texts)):
#     if not texts[i]:
#         count += 1
#         texts[i] = ''

# count

In [26]:
# for track in tracks:
#     if not track.get('n_tokens') and track.get('lyrics'):
#         track['n_tokens'] = len(track['lyrics'].split())
#     elif not track.get('n_tokens') and not track.get('lyrics'):
#         track['n_tokens'] = 0

In [27]:
# # fill meta fields with median
# import statistics

# def set_missing_to_median(field):
#     values = []

#     for track in tracks:
#         if track.get(field):
#             track[field] = float(track[field])
#             values.append(float(track[field]))

#     median = statistics.median(values)

#     for track in tracks:
#         if not track.get(field):
#             track[field] = median

In [28]:
# tracks[0].keys()

In [29]:
# import ast

# swears_it = set()
# swears_en = set()

# for track in tracks:
#     if track.get('swear_IT_words'):
#         track['swear_IT_words'] = ast.literal_eval(track['swear_IT_words'])
#         swears_it.update(track['swear_IT_words'])
#     if track.get('swear_EN_words'):
#         track['swear_EN_words'] = ast.literal_eval(track['swear_EN_words'])
#         swears_en.update(track['swear_EN_words'])

# len(swears_en), len(swears_it)

In [30]:
# for track in tracks:
#     if track.get('lyrics'):
#         tokenized_lyrics = track['lyrics'].strip().split()
#         curr_swears_en = []
#         curr_swears_it = []
#         for tok in tokenized_lyrics:
#             if tok in swears_en:
#                 curr_swears_en.append(tok)
#             if tok in swears_it:
#                 curr_swears_it.append(tok)
#         track['swear_EN_words'] = curr_swears_en
#         track['swear_EN'] = len(curr_swears_en)
#         track['swear_IT_words'] = curr_swears_it
#         track['swear_IT'] = len(curr_swears_it)
#         if len(curr_swears_it) > 0 or len(curr_swears_en) > 0:
#             track['explicit'] = True
#         else:
#             track['explicit'] = False
#     else:
#         track['lyrics'] = ''

In [31]:
# keys_median = [
#     'bpm', 'rolloff', 'flux', 'rms',
#     'flatness', 'spectral_complexity', 'pitch',
#     'loudness', 'duration_ms'
# ]

# for key in keys_median:
#     set_missing_to_median(key)

In [32]:
# for track in tracks:
#     if not track.get('streams@1month'):
#         track['streams@1month'] = 0
#     if not track.get('lyrics'):
#         track['lyrics'] = ''
#     if not track.get('popularity'):
#         track['popularity'] = 0

In [33]:
# for track in tracks:
#     if not track.get('year') and track.get('album_release_date'):
#         date = track['album_release_date']
#         date = date.strip().split('-')
#         track['year'] = date[0]
#         if len(date) > 1:
#             track['month'] = date[1]
#             track['day'] = date[2]

In [34]:
# missing_year = []
# missing_month = []

# for track in tracks:
#     if not track.get('year'):
#         missing_year.append(track)
#     if not track.get('month'):
#         missing_month.append(track)
        
# len(missing_month), len(missing_year)

In [35]:
# with open(f'{dataset_dir}/correct_ids/tracks_filled.json', 'w') as f:
#     json.dump(tracks, f)

In [36]:
# import numpy as np

# keys = [
#     'bpm', 'year', 'duration_ms',
#     'n_tokens', 'pitch', 'loudness'
# ]

In [37]:
# meta_arrays = [np.array([float(t[key]) for t in tracks]) for key in keys]


# meta = np.column_stack(meta_arrays)

In [38]:
my_stopwords = [
    'ah', 'uh', 'yeah', 'ehi', 'eh', 'seh', 'pe',
    'uoh', 'no', 'yah', 'mhm', 'oh', 'ca', 'nu',
    'int', 'you', 'the', 'to', 'it', 'and', 'my',
    'we', 'on', 'your', 'that', 'na', 'ra', 'ta',
    'lyrics', 'contributorsintro', 'contributorsil',
    'interludio', 'song', 'pt', 'pubblicata', 'that',
    'tha', 'contributoril', 'skrrt', 'nn', 'ohm',
    'lario', 'badabum', 'mcf', 'contributorsbloody',
    'aahhhh', 'pes', 'busdeez', 'lewa', 'amemì', 'llámame',
    'pih', 'baing', 'grah', 'ciny', 'lyricscoming',
    'lyricsthis', 'instrumental', 'contributorsinterlude',
    'eooh', 'phi', 'att'
]

In [39]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
ita_stopwords = stopwords.words('italian')

ita_stopwords.extend(my_stopwords)

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/brunobarbieri/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [40]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

nltk.download('stopwords')

vectorizer = TfidfVectorizer(max_df=0.95, min_df=2, stop_words=ita_stopwords)
dtm = vectorizer.fit_transform(texts)

num_topics = 8
nmf_model = NMF(n_components=num_topics, random_state=1)
nmf_model.fit(dtm)

topic_vectors = nmf_model.components_

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/brunobarbieri/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [41]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

W = nmf_model.transform(dtm)
# W = StandardScaler().fit_transform(W)

# PCA to topic vector
# pca_topics = PCA(n_components=12, random_state = 42)
# W = pca_topics.fit_transform(W)

# meta_scaled = StandardScaler().fit_transform(meta)

# W = np.hstack([W, meta_scaled])
H = nmf_model.components_

In [42]:
W

array([[0.00235826, 0.        , 0.04560645, ..., 0.00186559, 0.00493046,
        0.        ],
       [0.        , 0.00031144, 0.01603206, ..., 0.        , 0.00394421,
        0.        ],
       [0.        , 0.00112709, 0.04102779, ..., 0.02685986, 0.00312617,
        0.01368076],
       ...,
       [0.00033376, 0.00127672, 0.01736279, ..., 0.00049012, 0.01399782,
        0.        ],
       [0.01690318, 0.        , 0.03303907, ..., 0.        , 0.        ,
        0.00189709],
       [0.00556233, 0.        , 0.0315902 , ..., 0.        , 0.        ,
        0.        ]], shape=(11166, 8))

In [43]:
import numpy as np

n_top_words = 10
feature_names = vectorizer.get_feature_names_out()

for topic_idx, topic in enumerate(H):
    top_terms = [feature_names[i] for i in topic.argsort()[-n_top_words:][::-1]]
    print(f"Topic {topic_idx}: {', '.join(top_terms)}")


Topic 0: solo, mondo, ogni, ancora, vita, ora, quando, giorno, sempre, qui
Topic 1: nun, cchiù, pecché, ce, cu, mo, me, sempe, sulo, aggio
Topic 2: fra, cazzo, rap, tipo, frate, merda, fare, soldi, flow, fa
Topic 3: te, me, so, sai, baby, solo, vuoi, quando, cosa, voglio
Topic 4: bu, gang, milano, money, go, okay, santana, soldi, bang, squad
Topic 5: love, is, don, what, like, up, know, this, for, with
Topic 6: bene, va, sì, così, male, stare, so, dimmi, qui, poi
Topic 7: mai, sai, dire, guai, cosa, ora, vai, so, dimmi, fatto


In [44]:
import numpy as np

# 1. Hard assignment of songs to topics
W = nmf_model.transform(dtm)  # shape: (n_songs, n_topics)
topic_assignment = W.argmax(axis=1)  # each song -> main topic

feature_names = vectorizer.get_feature_names_out()
num_topics = nmf_model.n_components

# 2. Collect all song indices per topic
topic_docs = {i: [] for i in range(num_topics)}
for i, topic in enumerate(topic_assignment):
    topic_docs[topic].append(i)

# 3. Compute term frequency (TF) per topic
topic_tf = np.zeros((num_topics, len(feature_names)))

for topic, docs in topic_docs.items():
    if len(docs) == 0:
        continue
    topic_tf[topic, :] = dtm[docs, :].sum(axis=0)

# 4. Compute topic-level inverse document frequency (IDF)
term_in_topics = np.sum(topic_tf > 0, axis=0)  # how many topics each term appears in
topic_idf = np.log(num_topics / (1 + term_in_topics))  # add 1 to avoid div by zero

# 5. Compute topic-level TF-IDF
topic_tfidf = topic_tf * topic_idf[np.newaxis, :]

# 6. Get top words per topic
n_top_words = 10
topic_top_words = []

for topic_idx in range(num_topics):
    top_indices = topic_tfidf[topic_idx].argsort()[::-1][:n_top_words]
    top_words = [feature_names[i] for i in top_indices]
    topic_top_words.append(top_words)
    print(f"Topic {topic_idx}: {', '.join(top_words)}")

# 7. Optional: flag words appearing in many topics as candidate noise
threshold = int(num_topics)  # words appearing in 100% of topics
candidate_noise = [feature_names[i] for i, count in enumerate(term_in_topics) if count > threshold]
print("\nCandidate noise words:", candidate_noise)


Topic 0: renderò, xananas, disorientato, blocchiamo, ricordarti, odierai, mischiate, cercherà, dividerà, sembrerà
Topic 1: ammore, maje, primma, assaje, saccio, nzieme, ssaje, vedé, bbene, sulo
Topic 2: bvlgari, gengis, servile, arab, deng, mdsk, kompare, woop, chicoria, capetto
Topic 3: clonerò, nanananana, scalerò, scandalosa, richiudi, crisalide, sube, proverebbero, cosà, esmeralda
Topic 4: dpg, expensive, pshh, belt, ciapa, nonstop, pyrex, switchi, 223, wop
Topic 5: soon, contributorsoutro, enough, recognize, hand, talkin, coolin, even, won, somebody
Topic 6: suonino, ridimmi, modì, namora, waladi, uou, habibi, miniere, aumma, cacceresti
Topic 7: bussò, gostoso, accetterò, decidermi, ritornerò, fermerai, kobra, ridimensionati, maledirai, domai

Candidate noise words: []


In [45]:
# topic_assignment: each song -> its main topic
num_topics = nmf_model.n_components

# Count songs per topic
topic_counts = np.zeros(num_topics, dtype=int)

for t in range(num_topics):
    topic_counts[t] = np.sum(topic_assignment == t)

# Print counts
for t, count in enumerate(topic_counts):
    print(f"Topic {t}: {count} songs")


Topic 0: 3441 songs
Topic 1: 472 songs
Topic 2: 3063 songs
Topic 3: 1180 songs
Topic 4: 513 songs
Topic 5: 831 songs
Topic 6: 826 songs
Topic 7: 840 songs
